---
title: "Lecture 22: Sparse matrices: storage, basic operations, graphs"
author: "Jamie Haddock"
format: 
    revealjs:
        output-file: Lecture22_slides
        slide-number: true
        preview-links: auto
        logo: figs/hmc.png
        css: input/slides.css
        incremental: true
        smaller: true
        code-fold: true
        embed-resources: true
        html-math-method: mathjax
        slide-level: 3
    html: 
        output-file: Lecture22
        code-fold: true
        embed-resources: true
        code-links:
        - text: "Open in Colab"
          href: "https://colab.research.google.com/github/jamiehadd/Math-189M-NumericalLinearAlgebra/blob/main/Lecture22_Sparse_matrices_storage_basic_operations_graphs.ipynb"
          icon: "laptop"
    pdf:
        output-file: Lecture22
        documentclass: article
        toc: true
        number-sections: true
        geometry:
          - top=1in
          - left=1in
          - bottom=1in
          - right=1in
format-links: false
jupyter: julia-1.13
colab:
  gh-user: "jamiehadd"   
  gh-repo: "Math-189M-NumericalLinearAlgebra" 
filters: 
  - input/remove-pause.lua
  - colab
execute:
  echo: true
  eval: true
---

# Sparse matrices

- Data and operators (matrices) are often **sparse**, so it is an important focus in numerical linear algebra to develop methods for linear algebraic problems that are efficient on sparse matrices
  (their cost depends on the number of nonzeros, not the ambient size of the matrix).
- We will briefly see solving (triangular) sparse linear systems to see the sorts of ideas that get used.

## Sparse matrix storage

- **Coordinate form**: store the nonzeros as a list of triplets $(i, j, a_{ij})$
  - very simple, but not so good for operations (e.g., matrix-vector multiplication)
- Better alternatives are the **compressed sparse row (CSR)** format and its "transpose", the **compressed sparse column (CSC)** format

## CSR/C format

**CSR example.** Consider the $4 \times 4$ matrix
$$
A = \begin{bmatrix} 4.1 & 0 & 2.9 & 0 \\ 1.2 & -0.3 & 0 & -0.1 \\ 0 & 7.2 & 9.2 & 0 \\ 0 & 0 & 0 & 1.0 \end{bmatrix}.
$$
CSR stores $A$ as three arrays (indexed from 1, as in Julia):

- `nzval`: the nonzero values, in row-wise order
- `colval`: the column index of each entry of `nzval`
- `rowptr`: where each row starts in `nzval` (row $i$ occupies `rowptr[i]` through `rowptr[i+1]-1`)

---

For our $A$:

$$
\begin{aligned}
\texttt{nzval} &= \begin{bmatrix} 4.1 & 2.9 & 1.2 & -0.3 & -0.1 & 7.2 & 9.2 & 1.0 \end{bmatrix} \\
\texttt{colval} &= \begin{bmatrix} 1 & 3 & 1 & 2 & 4 & 2 & 3 & 4 \end{bmatrix} \\
\texttt{rowptr} &= \begin{bmatrix} 1 & 3 & 6 & 8 & 9 \end{bmatrix}
\end{aligned}
$$

`rowptr` has length $m+1$, and the total storage is proportional to the number of nonzeros.

In [11]:
mutable struct SparseMatrixCSR
    m :: Int                 # number of rows
    n :: Int                 # number of columns
    rowptr :: Vector{Int}
    colval :: Vector{Int}
    nzval :: Vector{Float64}
end

A = SparseMatrixCSR(4, 4,
        [1, 3, 6, 8, 9],
        [1, 3, 1, 2, 4, 2, 3, 4],
        [4.1, 2.9, 1.2, -0.3, -0.1, 7.2, 9.2, 1.0])

SparseMatrixCSR(4, 4, [1, 3, 6, 8, 9], [1, 3, 1, 2, 4, 2, 3, 4], [4.1, 2.9, 1.2, -0.3, -0.1, 7.2, 9.2, 1.0])

---

**CSC** is the same, except that the nonzeros are stored column by column, with arrays `colptr` and `rowval` in place of `rowptr` and `colval`.

::: {.callout-note title="Exercise"}
Write down $A$ in CSC format.
:::

. . .

**Solution.** Going down each column of $A$ in turn:

$$
\begin{aligned}
\texttt{nzval} &= \begin{bmatrix} 4.1 & 1.2 & -0.3 & 7.2 & 2.9 & 9.2 & -0.1 & 1.0 \end{bmatrix} \\
\texttt{rowval} &= \begin{bmatrix} 1 & 2 & 2 & 3 & 1 & 3 & 2 & 4 \end{bmatrix} \\
\texttt{colptr} &= \begin{bmatrix} 1 & 3 & 5 & 7 & 9 \end{bmatrix}
\end{aligned}
$$

### Exercise

For a matrix stored in CSR/CSC format, how do you calculate

- the number of nonzeros in each column/row?
- the total number of nonzeros in the matrix?

## Matvec

Let's consider how to calculate sparse matrix-vector multiplication $y = Ax$ for a matrix stored in CSR:

In [12]:
function csr_matvec(A::SparseMatrixCSR, x)
    y = zeros(A.m)
    for i in 1:A.m
        for k in A.rowptr[i]:A.rowptr[i+1]-1
            y[i] += A.nzval[k] * x[A.colval[k]]
        end
    end
    return y
end

csr_matvec (generic function with 1 method)

. . .

::: {.callout-note title="Exercise"}
Try writing code for matvec with $A^\top$, where $A$ is given in CSR format.
:::

---

**Solution.** Loop over the rows of $A$ as before, but now each stored entry $a_{ik}$ *adds into* entry `colval[k]` of the output, instead of being gathered into entry $i$:

In [13]:
function csr_transpose_matvec(A::SparseMatrixCSR, x)
    y = zeros(A.n)
    for i in 1:A.m
        for k in A.rowptr[i]:A.rowptr[i+1]-1
            y[A.colval[k]] += A.nzval[k] * x[i]
        end
    end
    return y
end

csr_transpose_matvec (generic function with 1 method)

## Matrix norms

- Many matrix norms can be computed efficiently for CSR/CSC sparse matrices
  - Example, the Frobenius norm: only the nonzeros contribute,
  $$\|A\|_F = \Big( \sum_{i,j} |a_{ij}|^2 \Big)^{1/2} = \Big( \sum_k |\texttt{nzval[k]}|^2 \Big)^{1/2},$$
  so the cost is proportional to the number of nonzeros.

In [14]:
frobenius_norm(A::SparseMatrixCSR) = sqrt(sum(abs2, A.nzval))

frobenius_norm (generic function with 1 method)

- A notable exception is the 2-norm, since it requires estimating a singular value of $A$.

## Relationship to graphs

- It is not just *how* sparse a matrix is, but also the **pattern** of sparsity
- Recall the definition of a **graph**: $G = (V, E)$, where $V$ is a collection of vertices (nodes) and $E$ is a collection of edges
  - **directed**: an edge $(u,v) \in E$ is a directed arc from $u$ to $v$
  - **undirected**: we identify $(u,v)$ with $(v,u)$ and draw the edge without an arrow

![](fig9_1_graphs.png){fig-align="center" width="85%"}

---

- A **path** from a vertex $u$ to a vertex $v$ is a sequence of edges $(x_0, x_1), (x_1, x_2), \dots, (x_{t-1}, x_t)$ so that $u = x_0$ and $v = x_t$
- A graph is **connected** if there is a path from any vertex to any other vertex

---

- A **tree** is a connected graph with no cycles (so there is a unique path between any two vertices)
- **Root**: one designated vertex of the tree, drawn at the top
- **Parent/child**: if $(u,v)$ is an edge and $u$ is closer to the root than $v$ (the unique path from $v$ to the root passes through $u$), then $u$ is the **parent** of $v$ and $v$ is the **child** of $u$
- **Leaf**: a vertex with no children
- **Ancestor/descendant**: a parent of a parent of $\dots$ of $v$ is an **ancestor** of $v$; a child of a child of $\dots$ of $v$ is a **descendant** of $v$

![](fig9_1_tree.png){fig-align="center" width="55%"}

### Sparsity pattern

The sparsity pattern of a square $n \times n$ matrix $A$ can be viewed as a directed graph on $n$ vertices, with a directed edge $(i,j)$ from vertex $i$ to vertex $j$ if and only if $a_{ji} \neq 0$.

![](fig9_2_matrix_graph.png){fig-align="center" width="90%"}

- Note: we won't record self-loops in our graph visualizations since the matrices we'll consider will often have nonzero diagonals.